# Bronze Layer — Source Blob Realtime → Bronze Volume (v1 — Manual)

Copies raw CSV files from the instructor-provided source blob storage into the Bronze Volume on Unity Catalog for **all realtime entities**, preserving the exact source directory structure.

Same pattern as `01_bronze_blob_charging_sessions.ipynb` from Day 3, extended to cover all realtime entities in one notebook.

---

### Source layout (instructor blob — `dataenggdailystorage`)

```
wasbs://source@dataenggdailystorage.blob.core.windows.net/
  └── realtime/
        ├── charging_sessions/
        │     └── YYYY/MM/DD/HH/
        │           └── sessions_YYYYMMDD_HHMM.csv
        │           └── charging_sessions_YYYYMMDD_HHMM.csv  ← variant name (Jun 19 & Jun 22 only)
        └── maintenance_events/
              └── YYYY/MM/DD/HH/
                    └── maintenance_YYYYMMDD_HHMM.csv
                    └── maintenance_events_YYYYMMDD_HHMM.csv ← variant name (Jun 19 & Jun 22 only)
```

> **Only 2 entities in realtime/.** Other EV data (energy_prices, weather, etc.) comes from the
> VoltGrid API — handled by the ADF v4 pipeline (Day 5), not from this blob.

> **Duplicate files on Jun 19 & Jun 22:** Two CSVs exist in the same hour partition (standard +
> variant naming). This notebook copies both. The Silver layer deduplicates by primary key.

### Bronze Volume target (Unity Catalog — mirrors source exactly)

```
/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/
  └── realtime/
        ├── charging_sessions/    YYYY/MM/DD/HH/  *.csv
        └── maintenance_events/   YYYY/MM/DD/HH/  *.csv
```

---

### Load modes — controlled by `LOAD_MODE` in Cell 2

| `LOAD_MODE` | What it copies |
|---|---|
| `full` | All files across all entities — use for first run |
| `incremental` | Only the specific `YYYY/MM/DD/HH` partition across all entities |

## Cell 1 — Authenticate to source blob storage

Reads storage account, container, and SAS token from Key Vault via `kv-ev-scope` Databricks secret scope. Nothing is hardcoded.

**Must run every session** — `spark.conf.set` registers SAS credentials for the `wasbs://` driver. This does not persist across cluster restarts.

In [ ]:
SCOPE = "kv-ev-scope"

STORAGE_ACCOUNT = dbutils.secrets.get(scope=SCOPE, key="source-storage-account")
CONTAINER       = dbutils.secrets.get(scope=SCOPE, key="source-container")
SAS_TOKEN       = dbutils.secrets.get(scope=SCOPE, key="source-sas-token")

spark.conf.set(
    f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SAS_TOKEN
)

SOURCE_ROOT = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"

print(f"Storage account : {STORAGE_ACCOUNT}")
print(f"Container       : {CONTAINER}")
print(f"SAS token       : [REDACTED]")
print(f"Source root     : {SOURCE_ROOT}")
print("Source blob authenticated — OK")

## Cell 2 — Set load mode, partition, and entity list

**This is the only cell you change between runs.**

- `LOAD_MODE = "full"` → copies everything under `realtime/<entity>/` for all entities
- `LOAD_MODE = "incremental"` → copies only `YYYY/MM/DD/HH` folder for all entities

`ENTITIES` lists every realtime subfolder under `realtime/` in the source blob.
Add or remove entity names here if the source blob adds new folders.

In [ ]:
LOAD_MODE = "incremental"   # "full" or "incremental"

LOAD_YEAR  = "2026"
LOAD_MONTH = "07"
LOAD_DAY   = "11"
LOAD_HOUR  = "06"

# Realtime entity folders present in the source blob under realtime/
# Only 2 entities — other EV data (energy_prices, weather, etc.) comes from VoltGrid API (Day 5 ADF)
ENTITIES = [
    "charging_sessions",
    "maintenance_events",
]

BRONZE_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"
BASE_SUBPATH  = "realtime"

# Build source/bronze path pairs per entity
entity_paths = []
for entity in ENTITIES:
    if LOAD_MODE == "full":
        src = f"{SOURCE_ROOT}/{BASE_SUBPATH}/{entity}/"
        dst = f"{BRONZE_VOLUME}/{BASE_SUBPATH}/{entity}/"
    else:
        partition = f"{LOAD_YEAR}/{LOAD_MONTH}/{LOAD_DAY}/{LOAD_HOUR}"
        src = f"{SOURCE_ROOT}/{BASE_SUBPATH}/{entity}/{partition}/"
        dst = f"{BRONZE_VOLUME}/{BASE_SUBPATH}/{entity}/{partition}/"
    entity_paths.append({"entity": entity, "source_path": src, "bronze_path": dst})

print(f"LOAD_MODE : {LOAD_MODE}")
if LOAD_MODE == "incremental":
    print(f"Partition : {LOAD_YEAR}/{LOAD_MONTH}/{LOAD_DAY}/{LOAD_HOUR}")
print(f"Entities  : {len(ENTITIES)}")
print()
for ep in entity_paths:
    print(f"  [{ep['entity']}]")
    print(f"    src : {ep['source_path']}")
    print(f"    dst : {ep['bronze_path']}")

## Cell 3 — List source files for all entities

Recursively lists all files at each entity source path. Use this to confirm paths are correct and expected files are visible before copying.

In [ ]:
def list_files_recursive(path):
    try:
        items = dbutils.fs.ls(path)
    except Exception:
        return []
    files = []
    for item in items:
        if item.isDir():
            files.extend(list_files_recursive(item.path))
        else:
            files.append(item)
    return files

total_source = 0
for ep in entity_paths:
    files = list_files_recursive(ep["source_path"])
    ep["source_files"] = files
    total_source += len(files)
    print(f"  {ep['entity']:<30} {len(files)} file(s)")
    for f in files:
        size_kb = round(f.size / 1024, 1)
        print(f"    {f.path.replace(ep['source_path'], '')}  [{size_kb} KB]")

print(f"\nTotal source files : {total_source}")

## Cell 4 — Copy files to Bronze Volume

Copies each file from the source blob to the Bronze Volume per entity, recreating the full directory structure.

- Relative path is extracted by stripping `source_path` prefix — this preserves folder structure
- `dbutils.fs.cp` creates intermediate folders automatically — no pre-creation needed
- Overwriting an existing file is safe — re-running is idempotent

In [ ]:
all_copied  = []
all_skipped = []

for ep in entity_paths:
    entity      = ep["entity"]
    source_path = ep["source_path"]
    bronze_path = ep["bronze_path"]
    source_files = ep["source_files"]

    if not source_files:
        print(f"  [{entity}] SKIP — no source files found")
        continue

    copied  = []
    skipped = []

    print(f"\n--- {entity} ({len(source_files)} file(s)) ---")
    for file_info in source_files:
        relative_path = file_info.path.replace(source_path, "")
        dest_path     = bronze_path + relative_path
        try:
            dbutils.fs.cp(file_info.path, dest_path)
            copied.append(dest_path)
            print(f"  COPIED  {relative_path}")
        except Exception as e:
            skipped.append((file_info.path, str(e)))
            print(f"  FAILED  {relative_path} — {e}")

    print(f"  Result: {len(copied)} copied, {len(skipped)} failed")
    all_copied.extend(copied)
    all_skipped.extend(skipped)

print(f"\nTotal: {len(all_copied)} copied, {len(all_skipped)} failed across {len(ENTITIES)} entities")

if all_skipped:
    raise Exception(f"{len(all_skipped)} file(s) failed to copy — check output above.")

## Cell 5 — Verify files in Bronze Volume

Lists files present in the Bronze Volume per entity and asserts count matches source. If any entity fails the assertion, the run is marked as failed.

In [ ]:
all_pass = True
total_bronze = 0

for ep in entity_paths:
    entity       = ep["entity"]
    bronze_path  = ep["bronze_path"]
    source_files = ep["source_files"]

    if not source_files:
        continue

    bronze_files = list_files_recursive(bronze_path)
    total_bronze += len(bronze_files)

    status = "PASS" if len(bronze_files) == len(source_files) else "FAIL"
    print(f"  [{status}] {entity:<30} source={len(source_files)}  bronze={len(bronze_files)}")

    if status == "FAIL":
        all_pass = False

print(f"\nTotal files in Bronze Volume : {total_bronze}")

if not all_pass:
    raise AssertionError("One or more entities have a count mismatch — check output above.")
else:
    print("All entities verified — OK")

## Cell 6 — Read a sample file per entity and inspect schema

Reads the first CSV from each entity's Bronze Volume path into a Spark DataFrame. Confirms files are readable and columns look as expected. Silver layer will enforce explicit types — `inferSchema` here is for inspection only.

In [ ]:
for ep in entity_paths:
    entity      = ep["entity"]
    bronze_path = ep["bronze_path"]

    bronze_files = list_files_recursive(bronze_path)
    if not bronze_files:
        print(f"  [{entity}] No files in Bronze — skipping schema check")
        continue

    sample = bronze_files[0].path
    print(f"\n--- {entity} ---")
    print(f"  Sample file : {sample.replace(bronze_path, '')}")

    df = spark.read.option("header", True).option("inferSchema", True).csv(sample)
    print(f"  Rows        : {df.count():,}")
    print(f"  Columns     : {len(df.columns)}")
    df.printSchema()

## Cell 7 — Run summary

Prints a final summary: load mode, partition, entity counts, file counts, failures.

In [ ]:
print("=" * 60)
print("BRONZE BLOB MIGRATION (ALL ENTITIES) — RUN SUMMARY")
print("=" * 60)
print(f"Load mode      : {LOAD_MODE}")
if LOAD_MODE == "incremental":
    print(f"Partition      : {LOAD_YEAR}/{LOAD_MONTH}/{LOAD_DAY}/{LOAD_HOUR}")
print(f"Entities       : {len(ENTITIES)}")
print(f"Files copied   : {len(all_copied)}")
print(f"Files failed   : {len(all_skipped)}")
print()
for ep in entity_paths:
    n = len(ep.get('source_files', []))
    status = "COPIED" if n > 0 else "SKIPPED (no source files)"
    print(f"  [{status}] {ep['entity']}  ({n} file(s))")
if all_skipped:
    print("\nFailed files:")
    for src, err in all_skipped:
        print(f"  {src}  →  {err}")
print("=" * 60)
print("Next step: Silver layer reads from Bronze Volume and writes Delta")